In [1]:
# import necessary packages
import xarray as xr
import numpy as np
import pandas as pd
import os
import glob

In [3]:
regions_seasons_dict = {
    'eastern_east_africa': {
        'OND': [10, 11, 12],
        'MAM': [3, 4, 5]
    },
    'lake_victoria_basin': {
        'DJF': [12, 1, 2],
        'MAM': [3, 4, 5],
        'SON': [9, 10, 11]
    },
    'west_africa': {
        'JAS': [7, 8, 9]
    },
    'southern_africa': {
        'DJF': [12, 1, 2],
        'FMA': [2, 3, 4]
    },
    'south_sudan': {
        'MJJ': [5, 6, 7],
        'JAS': [7, 8, 9],
        'ASO': [8, 9, 10]
    },
    'eastern_ukraine': {
        'DJF': [12, 1, 2],
        'MAM': [3, 4, 5],
        'AMJ': [4, 5, 6],
        'JA': [7, 8]
    },
    'sri_lanka': {
        'OND': [10, 11, 12]
    }
}


dict_keys(['OND'])

In [31]:
def convert_monthly_to_seasonal(file_path, regions_seasons_dict, save_path, start_year = 1993, end_year = 2024):
    """
    This function takes a merged monthly netcdf file and converts it to seasonal

    Arguments:
    - file_path: path to merged monthly netcdf file
    - regions_seasons_dict: dictionary of regions and their seasons
    - save_path: path to save the seasonal netcdf file

    Usage Notes:
    Ensure that the file_path follows this format:
    '/content/drive/MyDrive/data/netCDF/eastern_east_africa_CanESM5_merged.nc'
    It does not matter what the netcdf file name is, as long as it follows the format of
    some_region_here_model_merged.nc

    Ensure that the save_path follows this format:
    '/content/drive/MyDrive/data/netCDF'
    Again, it does not matter what the folder name is, as long as it does not end with a '/' or anything else after the folder name.

    Data Notes:
    The merged monthly netcdf file used have the following columns:
    - latitude
    - longitude
    - predicted_precip
    - actual_precip (from CHIRPS)
    - date
    - lead_time

    This merged data has been pre-processed with the monthly_merged_data_generation.py script.

    Regions and Seasons Notes:
    The regions_seasons_dict is a dictionary of regions, their seasons, and specific values of month minus lead time.
    Please refer to the above matrix of month minus lead time values to understand how these values were determined,
    as well as how the dictionary works.
    """

    # extract relevant information from the file path
    split = file_path.split('/') # split into list

    file_name = split[-1] # get the file name

    name_split = file_name.split('_') # get the name of the region, i.e [eastern, east, africa]

    region_name = '_'.join(name_split[0:-2]) # combine the name of the region, i.e eastern_east_africa

    new_file_name= file_name.replace('.nc', '_seasonal.csv') # make new file name for saving

    # status
    print(f'Converting {file_name} to seasonal...')

    # check if region name is in regions_seasons_dict
    if region_name not in regions_seasons_dict:
      print(f"ValueError: Region '{region_name}' not found in regions_seasons_dict.")
      return

    # open file
    merged_monthly_file = xr.open_dataset(file_path)

    # convert to a dataframe for pre-processing, ensemble mean
    merged_monthly_file_df = merged_monthly_file.to_dataframe().reset_index().dropna().groupby(['time', 'lead_time', 'latitude', 'longitude'])[['predicted_precip', 'precip']].mean().reset_index()

    # seperate month and year into seperate columns
    merged_monthly_file_df['month'] = merged_monthly_file_df['time'].dt.month
    merged_monthly_file_df['year'] = merged_monthly_file_df['time'].dt.year

    # subset to the time range of interest
    merged_monthly_file_df = merged_monthly_file_df.query('year >= 1993 and year <= 2024')

    # access the dictionary items of the given region
    season_data = regions_seasons_dict[region_name]

    # Iterate through seasons in the region
    for season_name, months in season_data.items():
      # Create a new column for the current season
      column_name = f"{season_name}"
      merged_monthly_file_df[column_name] = None  # initialize empty season column
      condition = merged_monthly_file_df['month'].isin(months)
      merged_monthly_file_df.loc[condition, column_name] = season_name

    season_dfs = [] # initialzie empty list of seasonal dataframes

    seasons = regions_seasons_dict[region_name].keys() # get the list of seasons for that region

    # for each season in season list, pivot the data into separate dataframes
    # if a region has 3 seasons,there will be 3 dataframes
    for season in seasons:
      if season in merged_monthly_file_df.columns:
          temp_df = merged_monthly_file_df[['lead_time', 'latitude', 'longitude',
                                'predicted_precip', 'precip', 'year', 'month', season]].copy()
          temp_df = temp_df.rename(columns={season: 'season'})
          temp_df = temp_df.dropna(subset=['season'])  # Drop rows where season is NaN
          season_dfs.append(temp_df)

    # Combine all seasonal rows
    long_format_df = pd.concat(season_dfs, ignore_index=True)

    # taking the spatial mean
    long_format_df = long_format_df.groupby(['lead_time', 'season', 'year', 'month'])[['predicted_precip', 'precip']].mean().reset_index()

    # extract current model from file path
    current_model = name_split[-2]

    # assign model name column
    long_format_df['model'] = str(current_model)
    long_format_df['region'] = str(region_name)

    # adding date of prediction back
    month_of_prediction = long_format_df['month'] - long_format_df['lead_time'].astype(int)
    long_format_df['year_of_prediction'] = long_format_df['year'] - (month_of_prediction < 1).astype(int)  # Adjust year if month < 1
    month_of_prediction[month_of_prediction <= 0] = month_of_prediction[month_of_prediction <= 0] + 12  # Adjust month to be between 1 and 12
    long_format_df['month_of_prediction'] = month_of_prediction

    long_format_df = long_format_df.rename(columns={'month': 'realization_month', 'year': 'realization_year', 'month_of_prediction': 'month', 'year_of_prediction': 'year'}) # rename columns to match datetime format
    long_format_df['date_of_prediction'] = pd.to_datetime(long_format_df[['year', 'month']].assign(day=1)) # create date of prediction column
    long_format_df = long_format_df[(long_format_df['realization_year'] >= start_year) & (long_format_df['realization_year'] <= end_year) ] # filter to start year
    long_format_df = long_format_df.drop(columns=['month', 'year', 'realization_month', 'lead_time']) # drop prediction columns

    # seasonal averages
    long_format_df = long_format_df.groupby(['region', 'model', 'season', 'date_of_prediction', 'realization_year'])[['predicted_precip', 'precip']].mean().reset_index()
    # save as csv
    new_save_path = save_path + '/' + new_file_name
    #long_format_df.to_csv(new_save_path)

    return long_format_df



In [32]:
file_name = 'data/netCDF/eastern_east_africa_DWD_merged.nc'

convert_monthly_to_seasonal(file_name, regions_seasons_dict, 'data/csv')

Converting eastern_east_africa_DWD_merged.nc to seasonal...


,lead_time,season,year,month,predicted_precip,precip,model,region
0,0.5,MAM,1993,3,0.057567,12.817122,DWD,eastern_east_africa
1,0.5,MAM,1993,4,1.102054,70.284767,DWD,eastern_east_africa
2,0.5,MAM,1993,5,1.827472,105.624176,DWD,eastern_east_africa
3,0.5,MAM,1994,3,0.207539,17.467175,DWD,eastern_east_africa
4,0.5,MAM,1994,4,0.750853,108.529701,DWD,eastern_east_africa
...,...,...,...,...,...,...,...,...
1113,5.5,OND,2023,11,1.567624,222.688187,DWD,eastern_east_africa
1114,5.5,OND,2023,12,0.316101,21.501926,DWD,eastern_east_africa
1115,5.5,OND,2024,10,1.358363,50.639233,DWD,eastern_east_africa
1116,5.5,OND,2024,11,0.977552,80.343330,DWD,eastern_east_africa


In [ ]:
# use glob to get all file names in netCDF
#file_paths = glob.glob('data/netCDF/*.nc')

# if \\ is in file path then replace with / so to match function requirements
file_paths = [f.replace("\\", "/") for f in glob.glob("data/netCDF/*") if os.path.isfile(f)]

for file_name in file_paths:
    convert_monthly_to_seasonal(file_name,
                                regions_seasons_dict=regions_seasons_dict,
                                save_path='data/csv')